# Day 5: Session 5A - The Split-Apply-Combine Pattern

[Session Webpage](https://eds-217-essential-python.github.io/course-materials/interactive-sessions/5a_grouping_data.html)

Date: 09/04/2026

In [1]:
import pandas as pd

url = "https://eds-217-essential-python.github.io/data/messy_field_survey.csv"
survey = pd.read_csv(url)

survey = survey.drop_duplicates()

survey["site"] = survey["site"].str.strip().str.lower().str.replace("-", "_")
# survey['site'] = survey['site'].str.lower()
# survey['site'] = survey['site'].str.replace('-', '_')

survey["pH"] = survey["pH"].str.replace(",", ".").astype(float)
# survey['pH'] = survey['pH'].astype(float)

survey = survey.dropna(
    subset=["temperature_c", "dissolved_oxygen_mg_L", "conductivity_uS_cm"]
)
survey["n_replicates"] = survey["n_replicates"].fillna(1).astype(int)
#survey["n_replicates"] = survey["n_replicates"].astype(int)

survey = survey[survey["temperature_c"] > -100].copy()
survey = survey.rename(columns={"collection date": "collection_date"})

survey.shape

(255, 7)

In [2]:
survey['dissolved_oxygen_mg_L'].mean()

8.062039215686275

### describe what .groupby() does in terms of split, apply, combine


In [3]:
site_d = survey[survey['site'] == 'site_d']
site_f = survey[survey['site'] == 'site_f']

print(site_d['dissolved_oxygen_mg_L'].mean())
print(site_f['dissolved_oxygen_mg_L'].mean())
# this is taxing and pedantic
# let's groupy by site and calc mean from there

10.053333333333333
6.180232558139535


In [4]:
grouped = survey.groupby('site')

In [5]:
type(grouped)

pandas.core.groupby.generic.DataFrameGroupBy

In [ ]:
grouped['dissolved_oxygen_mg_L'].mean()
# indices become site labels, or whatever you grouped by

site
site_a     9.485909
site_b     8.601220
site_c     6.873111
site_d    10.053333
site_e     7.410930
site_f     6.180233
Name: dissolved_oxygen_mg_L, dtype: float64


### write the split-apply-combine pattern in its two-step form and its one-line form, and explain why they are the same pattern


In [9]:
## Two Step Form
# Step 1. Make groupby object
grouped = survey.groupby('site')

# Step 2. Aggregate on the object for a specific column
print(grouped['dissolved_oxygen_mg_L'].mean())

## One Step Form
# Groupby and aggreagate on that
grouped = survey.groupby('site')['dissolved_oxygen_mg_L'].mean()
print(grouped)
print("Same results !")

site
site_a     9.485909
site_b     8.601220
site_c     6.873111
site_d    10.053333
site_e     7.410930
site_f     6.180233
Name: dissolved_oxygen_mg_L, dtype: float64
site
site_a     9.485909
site_b     8.601220
site_c     6.873111
site_d    10.053333
site_e     7.410930
site_f     6.180233
Name: dissolved_oxygen_mg_L, dtype: float64
Same results !


In [ ]:
## Split-pick-apply pattern
## df.groupby('key')['column'].aggregation()
###           split      pick        apply
#groupby('key') says which piles to make. 
# The key is the column whose values name the groups.
# ['column'] says which column to do analysis on.
# .aggregation() says what arithmetic.

### read the result of a grouped calculation and explain what its index means


In [13]:
mean_do = survey.groupby('site')['dissolved_oxygen_mg_L'].mean()
# code left to right, read right to left
# "taking the mean of dissolved oxygen by site in survey dataset"

print(type(mean_do))
print(mean_do.index)
print("taking the mean of dissolved oxygen by site in survey dataset")

<class 'pandas.core.series.Series'>
Index(['site_a', 'site_b', 'site_c', 'site_d', 'site_e', 'site_f'], dtype='object', name='site')
taking the mean of dissolved oxygen by site in survey dataset


In [11]:
mean_do.idxmax()

'site_d'

In [12]:
print('Best site:', mean_do.idxmax())
print('Worst site:', mean_do.idxmin())

Best site: site_d
Worst site: site_f


In [20]:
# Practice
temp_group = survey.groupby('site')['temperature_c'].mean()
temp_group.idxmax()

'site_f'


### learn to choose aggregations based on the question you need to answer


.mean()	: what is typical in each group

.median() : what is typical, when a few extreme values would drag the mean

.sum() : how much in total, per group

.count() : how many rows went into each answer

.min(), .max() : the extremes within each group

.std() : how spread out each group is

In [21]:
survey.groupby('site')['dissolved_oxygen_mg_L'].max()

site
site_a    11.12
site_b     9.86
site_c     7.99
site_d    11.43
site_e     8.79
site_f     7.77
Name: dissolved_oxygen_mg_L, dtype: float64

In [22]:
survey.groupby('site')['dissolved_oxygen_mg_L'].min()

site
site_a    8.21
site_b    7.33
site_c    5.65
site_d    8.60
site_e    5.16
site_f    4.14
Name: dissolved_oxygen_mg_L, dtype: float64

In [23]:
survey.groupby('site')['n_replicates'].sum()

site
site_a    130
site_b    115
site_c    131
site_d    116
site_e    127
site_f    127
Name: n_replicates, dtype: int64

In [ ]:
survey.groupby('site')['dissolved_oxygen_mg_L'].count()
# .count() is not an afterthought
# A group mean computed from four rows and a group mean computed 
# from four hundred are printed in exactly the same font. 
# Nothing in the output warns you. 
# Whenever you report a grouped mean, run .count() on the same grouping 
# and look at it, even if it never reaches your report.

site
site_a    44
site_b    41
site_c    45
site_d    39
site_e    43
site_f    43
Name: dissolved_oxygen_mg_L, dtype: int64

In [ ]:
# Practice: What is the highest pH recorded at each site?
pH_group = survey.groupby('site')['pH'].max()
pH_group

site
site_a    7.92
site_b    7.41
site_c    7.10
site_d    8.33
site_e    7.47
site_f    6.98
Name: pH, dtype: float64

In [ ]:
# How many bottles were filled in total at each site? (n_replicates counts bottles.)
bottles_group = survey.groupby('site')['n_replicates'].sum()
bottles_group
# Use .sum() rather than .count() to add cells together, not count cells

site
site_a    130
site_b    115
site_c    131
site_d    116
site_e    127
site_f    127
Name: n_replicates, dtype: int64

### explain what can be a key

In [32]:
# Any column whose values repeat can be a grouping key
# The values do not have to be text, and they do not have to be tidy, 
# but they do have to mean something

# pH has 138 different values in the dataset, so let's bin them into larger categories

def classify_ph(value):
    """Label a pH value as acidic, neutral, or alkaline."""
    if value < 6.5:
        return 'acidic'
    elif value > 7.5:
        return 'alkaline'
    else:
        return 'neutral'


survey['ph_class'] = survey['pH'].apply(classify_ph)

In [34]:
# Now we can use .groupby

survey.groupby('ph_class')['dissolved_oxygen_mg_L'].mean()
# Many of the most useful groupings you will make are ones you build 
# rather than ones the file hands you.

ph_class
acidic      6.350000
alkaline    9.948571
neutral     8.019181
Name: dissolved_oxygen_mg_L, dtype: float64


### explain why a grouped answer is often more useful than a whole-column answer

In [41]:
print("Dissolved O counts by", survey.groupby('site')['dissolved_oxygen_mg_L'].count())
print("Average dissolved O by", survey.groupby('site')['dissolved_oxygen_mg_L'].mean())
print("Average temperature O by", survey.groupby('site')['temperature_c'].mean())
print(
    "The sites with the warmest water have the least oxygen in it.")
print(
    "Warm water holds less dissolved oxygen than cold water.")

Dissolved O counts by site
site_a    44
site_b    41
site_c    45
site_d    39
site_e    43
site_f    43
Name: dissolved_oxygen_mg_L, dtype: int64
Average dissolved O by site
site_a     9.485909
site_b     8.601220
site_c     6.873111
site_d    10.053333
site_e     7.410930
site_f     6.180233
Name: dissolved_oxygen_mg_L, dtype: float64
Average temperature O by site
site_a    16.600000
site_b    18.102439
site_c    21.911111
site_d    15.179487
site_e    19.730233
site_f    23.297674
Name: temperature_c, dtype: float64
The sites with the warmest water have the least oxygen in it.
Warm water holds less dissolved oxygen than cold water.


## Key points
Split, apply, combine: .groupby() splits the table into piles, an aggregation runs on each pile, and the answers come back as one labelled result.

The split-apply-combine pattern is df.groupby('key')['column'].aggregation().

The two-step form and the one-line form are the same pattern. The two-step version keeps the split in a variable so you can reuse it.

.groupby() on its own calculates nothing. The arithmetic happens when you name an aggregation.

The result is a Series indexed by the group labels, so .idxmax(), .idxmin() and label lookup all still work.

Any aggregation you can run on a column can run on a group: .mean(), .sum(), .count(), .min(), .max(), .median(), .std().

Always look at .count() alongside a grouped mean. Unequal groups are invisible otherwise.
Grouping keys are often columns you derived, not columns you were given.